# CellSight — Cellpose-SAM on free Colab GPU

Runs the **foundation-model** half of CellSight (the part that wants a GPU) on Colab's free T4.
The custom OpenCV pipeline, featurization, and evaluation live in the repo and run on your laptop.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# 1. Install deps (torch is preinstalled on Colab; add cellpose = Cellpose-SAM)
!pip -q install cellpose scikit-image opencv-python-headless
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Get the CellSight code + dataset.
#    Option A: clone your GitHub repo (recommended once you've pushed it).
# !git clone https://github.com/<you>/CellSight.git && cd CellSight
#
#    Option B: pull the DSB dataset from Kaggle (needs kaggle.json token).
# from google.colab import files; files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c data-science-bowl-2018 -f stage1_train.zip
# !unzip -q stage1_train.zip -d data/stage1_train

In [ ]:
# 3. Zero-shot Cellpose-SAM inference on a few tiles.
import numpy as np, cv2, glob, os
from cellpose import models
from skimage.measure import regionprops_table
import pandas as pd

model = models.CellposeModel(gpu=True)   # Cellpose v4 = Cellpose-SAM

def read_gray(p):
    im = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if im.ndim == 3: im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    if im.dtype != np.uint8: im = cv2.normalize(im, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return im

paths = sorted(glob.glob('data/stage1_train/*/images/*.png'))[:5]
for p in paths:
    img = read_gray(p)
    masks, flows, styles = model.eval(img, diameter=None, flow_threshold=0.4)[:3]
    n = int(masks.max())
    print(os.path.basename(p), '->', n, 'cells')

In [ ]:
# 4. Featurize one result (per-cell morphology) and save masks back to disk.
img = read_gray(paths[0])
masks = model.eval(img, diameter=None)[0].astype(np.int32)
props = regionprops_table(masks, intensity_image=img,
    properties=['label','area','perimeter','eccentricity','solidity','mean_intensity'])
df = pd.DataFrame(props); print('cells:', len(df)); df.head()

## (Stretch) Light fine-tune

Cellpose-SAM zero-shot is already strong, so fine-tuning is optional. If you have time,
the Cellpose training API lets you refine on a handful of your own labelled tiles:

```python
from cellpose import train
# train.train_seg(model.net, train_data=[imgs], train_labels=[labels],
#                 n_epochs=50, learning_rate=1e-5, weight_decay=0.1,
#                 model_name='cellsight_finetuned')
```

Then re-run evaluation (`src/evaluate.py`) to show the fine-tuned model beats zero-shot.